# Rerun a stormfork experiment

This notebook reruns one published fork experiment end to end and checks your
rerun against the published tracks: initial conditions, the edit, the baseline,
fork and control rollouts, storm tracking, export, and a final comparison.

Pick an `experiment_id` from the list in step 2, set your GitHub login, and run
the cells in order. Hindcasts and sweeps are not covered here.

The last cell prints one JSON entry. For a published experiment, add it to
`site/public/experiments/<id>/reproductions.json` in a pull request (use the
"reproduction report" issue template). For a recipe you wrote yourself, upload
the zip from step 8 to the Hugging Face dataset instead.

## 1. Install and download the inputs

`stormfork fetch` lists the size of each input before it starts, then
downloads the model weights, the HRES sample and the IBTrACS best tracks once,
checks each pinned file's SHA-256 and prints the attribution each one needs.
The commands below read them from that cache.

In [ ]:
!git clone --depth 1 https://github.com/intesarjawad/stormfork.git
%cd stormfork
%pip install --quiet . "weathernext @ git+https://github.com/google-deepmind/weathernext.git@v0.3.0" "jax==0.12.0.1"

In [ ]:
!stormfork fetch

## 2. Pick the experiment

The list shows every experiment, its kind, its initial-condition source and its
init time. Only a `fork` can be rerun here.

In [ ]:
!stormfork scenarios list

In [ ]:
EXPERIMENT_ID = "milton_noise_1pct"
GITHUB_USER = "your-github-login"
RUNTIME = "cpu"  # cpu, gpu or tpu

In [ ]:
import os
import shlex
from pathlib import Path

from stormfork.cli_perturb import moved_centre
from stormfork.downloads import IBTRACS, cache_directory
from stormfork.perturb import Taper
from stormfork.scenarios import load_scenarios

scenario = next(item for item in load_scenarios() if item.id == EXPERIMENT_ID)
if scenario.kind != "fork" or scenario.edit is None:
    msg = f"{EXPERIMENT_ID} is a {scenario.kind}; this notebook reruns forks"
    raise ValueError(msg)

INIT = scenario.init[0].isoformat()
RUN_DIR = Path("runs") / EXPERIMENT_ID
IBTRACS_FILE = cache_directory(os.environ) / IBTRACS.filename
TARGET_ARGS = f"--target-track-id {scenario.storm}"

params = scenario.edit.params
centre, radius_km = (0.0, 0.0), 400.0
if scenario.taper is not None:
    centre, radius_km = scenario.taper.centre, scenario.taper.radius_km
elif scenario.edit.type == "sst_anomaly":
    centre = (float(params["centre"][0]), float(params["centre"][1]))
perturb = ["--operation", scenario.edit.type]
perturb += ["--centre-lat", str(centre[0]), "--centre-lon", str(centre[1])]
perturb += ["--radius-km", str(radius_km)]
for name, value in params.items():
    if name != "centre":
        perturb += [f"--{name.replace('_', '-')}", str(value)]
PERTURB_ARGS = shlex.join(perturb)

FORK_TRACK_ARGS = TARGET_ARGS
if scenario.edit.type == "relocate":
    moved = moved_centre(Taper(*centre, radius_km), params["east_km"], params["north_km"])
    FORK_TRACK_ARGS += f" --moved-lat {moved[0]} --moved-lon {moved[1]}"

## 3. Build the initial conditions

Writes the model input for the experiment's init time to `runs/<id>/input.nc`.

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
!stormfork ic build $EXPERIMENT_ID --init $INIT --out $RUN_DIR/input.nc

## 4. Check the edit

A dry run of the experiment's edit on that input. It prints statistics and
fails if the edit leaves the allowed range; it writes nothing.

In [ ]:
!stormfork perturb $RUN_DIR/input.nc --dry-run $PERTURB_ARGS

## 5. Roll out the forecasts

One command runs the three rollouts, baseline, fork and control, into
`runs/<id>/baseline`, `fork` and `control`, each with a `forecast.nc` and
member 0's maps, and writes the run's `metadata.json`. It records the storm's
position at the start time from the cached IBTrACS file. The rollout is the
slow step.

In [ ]:
!stormfork run $EXPERIMENT_ID --runtime $RUNTIME --out $RUN_DIR

## 6. Track the storm

The tracker reads the same IBTrACS file. The fork run is matched to the storm's
relocated position when the edit moves it.

In [ ]:
!stormfork track $RUN_DIR/baseline/forecast.nc --ibtracs $IBTRACS_FILE --init $INIT --out $RUN_DIR/baseline/tracks.csv $TARGET_ARGS
!stormfork track $RUN_DIR/fork/forecast.nc --ibtracs $IBTRACS_FILE --init $INIT --out $RUN_DIR/fork/tracks.csv $FORK_TRACK_ARGS
!stormfork track $RUN_DIR/control/forecast.nc --ibtracs $IBTRACS_FILE --init $INIT --out $RUN_DIR/control/tracks.csv $TARGET_ARGS

## 7. Export and print the hashes

Writes the manifest, tracks and divergence into the run directory, then prints
the SHA-256 of every file the manifest names.

In [ ]:
!stormfork export $RUN_DIR --out $RUN_DIR

In [ ]:
import hashlib
import json

manifest = json.loads((RUN_DIR / "manifest.json").read_text())
PUBLISHED_FILES = [
    "manifest.json",
    manifest["divergence"]["path"],
    *(artifact["path"] for artifact in manifest["tracks"].values()),
    *(path for assets in manifest["runs"].values() for path in assets),
]
for path in PUBLISHED_FILES:
    print(hashlib.sha256((RUN_DIR / path).read_bytes()).hexdigest(), path)

## 8. Zip for download

The zip holds only the files the manifest names, which is what the Hugging Face
upload for a new experiment expects:
`hf upload <owner>/<name>-experiments <zip> --repo-type dataset --create-pr`.

In [ ]:
import zipfile

ARCHIVE = Path(f"{EXPERIMENT_ID}.zip")
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in PUBLISHED_FILES:
        bundle.write(RUN_DIR / path, path)
print(ARCHIVE, ARCHIVE.stat().st_size, "bytes")
try:
    from google.colab import files

    files.download(str(ARCHIVE))
except ImportError:
    pass

## 9. Compare with the published run

Compares your ensemble-mean and member tracks with the published ones against
the runtime tolerance in `docs/reproducibility.md` and prints one JSON entry:
`by`, `runtime`, `date`, `code_version`, `max_track_diff_km` and `passed`.

In [ ]:
!stormfork verify $EXPERIMENT_ID --reproduction $RUN_DIR --by $GITHUB_USER